# 04-b  Fatigue Connectome Baselines — Hardening Audit

**Purpose:** Adversarial analyses that challenge the baseline findings
before any fatigue-related connectomic signal can be claimed.
This notebook is the **technical appendix** to `04_a`.
The two notebooks share a single authoritative verdict: **INDETERMINATE**.

| # | Analysis | Target | Estimand |
|---|----------|--------|----------|
| 1 | Max-permutation correction (7 channels) | D + E | Within-fold PCA — same as main run |
| 2 | Holm / BH corrections on main-run *p*-values | D + E | Within-fold PCA (main run) |
| 3 | Incremental value over metadata | D only | Within-fold PCA |
| 4 | Sex-matched subsampling (females) | D only | Within-fold PCA / LOOCV |
| 5 | Metadata decomposition (Age / Sex / Age+Sex) | D only | Within-fold CV |
| 6 | Power / precision analysis | D only | Bootstrap approximation |

> **Estimand note:** All AUCs and p-values in this notebook use
> **within-fold PCA** — the same procedure as the main-run pipeline in
> `04_a`.  Residualization, scaling, and PCA are each fitted on the
> training fold only.  This ensures the max-perm corrected p-values refer
> to the same estimand as the BH/Holm corrections.


In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 200,
                      "font.size": 10, "axes.titlesize": 12})

import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUT = PROJECT_ROOT / "results" / "fatigue_connectome_baselines"
FIG = OUT / "Figures"; FIG.mkdir(exist_ok=True)

def _load_json(name):
    p = OUT / "Tables" / name
    return json.loads(p.read_text()) if p.exists() else {}


---
## §1  Max-permutation correction across 7 channels

For each of 1 000 permuted label vectors, the **full within-fold pipeline**
(residualization → StandardScaler → PCA → LogisticRegression) is trained
on all 7 channels and the **maximum AUC** across channels is recorded.
The corrected *p*-value = fraction of null maxima ≥ observed best AUC.

**Estimand consistency:** Because PCA is fitted on the *training fold*
only (not globally), the observed AUCs here match those in the BH/Holm
correction table in §2.  The max-perm corrected p-value therefore tests
the same pipeline that produced the AUCs in the BH table.

> The label-free steps (residualization, scaling, PCA) are precomputed
> per fold once and only LogisticRegression is re-fitted per permutation,
> since it is the only label-dependent step.  This makes the test
> computationally tractable while preserving exact estimand consistency.


In [ ]:
mp_d = _load_json("hardening_D_max_perm.json")
mp_e = _load_json("hardening_E_max_perm.json")

null_d = np.load(OUT / "Tables" / "hardening_D_null_distributions.npz")
null_e = np.load(OUT / "Tables" / "hardening_E_null_distributions.npz")

estimand_label = mp_d.get("estimand", "within_fold_pca")

# Summary table — AUCs now match the BH table (same within-fold estimand)
rows = []
for tgt, mp in [("D", mp_d), ("E", mp_e)]:
    for cond in ["raw", "resid"]:
        key_best = f"best_{cond}_ch"
        key_auc  = f"best_{cond}_auc"
        rows.append(dict(
            Target=tgt, Condition=cond,
            Best_Channel=mp.get(key_best, "—"),
            AUC_withinFoldPCA=mp.get(key_auc, "—"),
            Corrected_p=mp.get(f"corrected_p_{cond}", "—"),
        ))
df_mp = pd.DataFrame(rows)
print(f"Estimand: {estimand_label}")
print(df_mp.to_string(index=False))
print()
print("Criterion: corrected_p < 0.05 required for FWER-level evidence.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, tgt, null, mp, label in [
    (axes[0], "D", null_d, mp_d, "Target D (COVID-only, n=53)"),
    (axes[1], "E", null_e, mp_e, "Target E (all groups, n=85)"),
]:
    for cond, color in [("raw", "#2196F3"), ("resid", "#FF5722")]:
        null_arr = null[f"null_max_{cond}"]
        obs = mp[f"best_{cond}_auc"]
        corr_p = mp[f"corrected_p_{cond}"]
        ax.hist(null_arr, bins=40, alpha=0.45, color=color,
                label=f"{cond}  obs={obs:.3f}  p_corr={corr_p:.3f}")
        ax.axvline(obs, color=color, ls="--", lw=2)
    ax.set_xlabel("Max AUC across 7 channels (null distribution)")
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.axvline(0.5, color="grey", ls=":", lw=0.8)
axes[0].set_ylabel("Frequency")
fig.suptitle("Max-permutation null distributions (within-fold PCA estimand)",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "fig6_max_perm_null_distributions.png", bbox_inches="tight")
fig.savefig(FIG / "fig6_max_perm_null_distributions.pdf", bbox_inches="tight")
plt.show()
print("Fig 6 saved.")


---
## §2  Holm–Bonferroni / Benjamini–Hochberg corrections

Applied to the **main-run within-fold permutation *p*-values** (1 000 perms,
same within-fold PCA estimand as §1).  7 channels tested per target × condition.

- **Holm**: step-down FWER control (α = 0.05)
- **BH**: step-up FDR control (q = 0.05)

### Interpretation of the BH result for Target D residualized

Two channels (MI\_KNN and dFC\_AbsDiffMean) share the same BH
q-value ≈ 0.038 for Target D residualized.

**This is not two independent FDR discoveries.**  The BH procedure
assigns q-values based on the joint rank structure of the full set of
7 p-values.  When two channels tie at the same rank position, they
receive the same q-value by construction.  The shared q reflects a
single rank-coupled outcome — one reason the FDR signal remains
exploratory rather than confirmatory:

- The two channels are likely correlated (both capture non-linear
  dynamic FC properties), so their p-values are not independent.
- A joint q ≈ 0.038 from two correlated tests at the same rank is
  weaker evidence than two independent FDR discoveries would be.
- BH control only survives here; the stronger Holm / FWER test
  does not (Holm p ≈ 0.070 for both channels).

**Conclusion for §2:** BH survival at q < 0.05 is exploratory only.
It does not constitute confirmatory evidence per the criterion hierarchy
in `04_a` (which requires max-perm p < 0.05 for CAUTIOUS GO).


In [ ]:
corr = pd.read_csv(OUT / "Tables" / "hardening_holm_bh_corrections.csv")

# Show only resid rows for clarity (main interest)
resid = corr[corr["condition"] == "resid"].copy()
resid = resid.sort_values(["target", "nominal_p"])
print("=== Residualised channels — corrections ===")
print(resid.to_string(index=False))
print()
print("Key:  Holm < 0.05 → survives FWER.  BH < 0.05 → survives FDR (exploratory).")


In [ ]:
# Paired bar chart: nominal vs Holm vs BH for Target D resid
d_resid = corr[(corr["target"] == "D") & (corr["condition"] == "resid")].sort_values("nominal_p")

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(d_resid))
w = 0.25
ax.bar(x - w, d_resid["nominal_p"], w, label="Nominal", color="#42A5F5")
ax.bar(x,     d_resid["holm_p"],    w, label="Holm (FWER)", color="#FF7043")
ax.bar(x + w, d_resid["bh_fdr_q"],  w, label="BH (FDR)", color="#66BB6A")
ax.axhline(0.05, color="red", ls="--", lw=1, label="α / q = 0.05")
ax.set_xticks(x)
ax.set_xticklabels(d_resid["channel"], rotation=35, ha="right")
ax.set_ylabel("p-value / q-value")
ax.set_title("Target D — Residualised: Nominal vs Corrected p-values\n"
             "(Note: MI_KNN and dFC_AbsDiffMean share the same BH q — rank-coupled)")
ax.legend(fontsize=8)
ax.set_ylim(0, min(1.0, d_resid["holm_p"].max() * 1.3))
fig.tight_layout()
fig.savefig(FIG / "fig7_holm_bh_corrections_D.png", bbox_inches="tight")
fig.savefig(FIG / "fig7_holm_bh_corrections_D.pdf", bbox_inches="tight")
plt.show()
print("Fig 7 saved.")


---
## §3  Incremental value over metadata  (Target D)

Does the connectome add predictive value beyond Age + Sex?

Three models (same 5-fold CV, within-fold PCA):
1. **Metadata only** (Age + Sex → StandardScaler → LogReg)
2. **Connectome only** (residualised → PCA(20) → LogReg)
3. **Combined** (resid PCA scores + Age + Sex → LogReg)

Paired bootstrap test on OOF predictions (2 000 resamples).

**Power caveat:** The wide confidence interval on ΔAUC reflects low
precision at n = 53, not necessarily an absent effect.
See §6 for a quantified power analysis of this result.


In [ ]:
iv_mi  = _load_json("hardening_D_incremental_value.json")
iv_dfc = _load_json("hardening_D_incremental_value_ch3.json")

print("=== Incremental Value — Target D ===")
print()
for lbl, iv in [("MI_KNN (ch2)", iv_mi), ("dFC_AbsDiffMean (ch3)", iv_dfc)]:
    if not iv:
        continue
    print(f"Channel: {lbl}")
    print(f"  Metadata-only AUC:   {iv['auc_metadata']:.4f}")
    print(f"  Connectome-only AUC: {iv['auc_connectome_resid']:.4f}  95%CI {iv['connectome_ci']}")
    print(f"  Combined AUC:        {iv['auc_combined']:.4f}")
    print(f"  ΔAUC (combined-meta): {iv['delta_comb_vs_meta']:.4f}  p={iv['delta_p']:.3f}  95%CI {iv['delta_ci']}")
    if iv['delta_p'] < 0.05:
        print("  → Connectome adds significant incremental value.")
    else:
        print("  → Not significant; wide CI — interpretation is power-limited.")
    print()


In [ ]:
# Bar chart: metadata vs connectome vs combined
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, lbl, iv in [
    (axes[0], "MI_KNN (ch2)", iv_mi),
    (axes[1], "dFC_AbsDiffMean (ch3)", iv_dfc),
]:
    if not iv:
        continue
    aucs = [iv["auc_metadata"], iv["auc_connectome_resid"], iv["auc_combined"]]
    labels = ["Metadata\n(Age+Sex)", f"Connectome\n(resid {lbl.split()[0]})", "Combined"]
    colors = ["#78909C", "#42A5F5", "#66BB6A"]
    bars = ax.bar(labels, aucs, color=colors, edgecolor="black", linewidth=0.5)
    ax.axhline(0.5, color="grey", ls=":", lw=0.8)
    for b, v in zip(bars, aucs):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.3f}",
                ha="center", fontsize=9)
    ax.set_ylim(0.4, 0.85)
    ax.set_ylabel("OOF AUC")
    delta = iv["delta_comb_vs_meta"]
    p = iv["delta_p"]
    sig_str = "*" if p < 0.05 else "n.s."
    ax.set_title(f"{lbl}\nΔAUC = {delta:+.3f}, p = {p:.3f} ({sig_str})")
fig.suptitle("Target D — Incremental Value of Connectome over Metadata\n"
             "(non-significant result is power-limited; see §6)", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "fig8_incremental_value.png", bbox_inches="tight")
fig.savefig(FIG / "fig8_incremental_value.pdf", bbox_inches="tight")
plt.show()
print("Fig 8 saved.")


---
## §4  Sex-matched subsampling  (Target D, females)

Target D has a significant sex imbalance (23F/6M in FATIGA EXTREMA vs 11F/13M
in NO HAY FATIGA, χ² = 5.03, p = 0.025).  Does the signal persist when sex is
held constant?

**Procedure:** Restrict to females (23 pos / 11 neg).
Subsample 11 from 23 positives → balanced 11 vs 11.
LOOCV with PCA(10) + LogReg on each subsample.  200 repetitions.

**Important caveat:** This analysis is a **descriptive sensitivity check**,
not a definitive falsification.  With n = 22 per sample and balanced
11 vs 11, the study is severely underpowered: even a true AUC = 0.70
would yield < 40% power at α = 0.05.  A near-chance result here means
*unconfirmed within-sex*, not *absent*.  See §6 for quantified precision.


In [ ]:
fem = pd.read_csv(OUT / "Tables" / "hardening_D_female_subsampling.csv")
male_path = OUT / "Tables" / "hardening_D_male_subsampling.csv"
male = pd.read_csv(male_path) if male_path.exists() else pd.DataFrame()

print("=== Female sex-matched subsampling (n≈22 per sample) ===")
for ch_name in fem["channel_name"].unique():
    aucs = fem.loc[fem["channel_name"] == ch_name, "auc"].dropna()
    print(f"  {ch_name:20s}  median={aucs.median():.3f}  "
          f"IQR=[{aucs.quantile(0.25):.3f}, {aucs.quantile(0.75):.3f}]  "
          f">0.5: {100*(aucs>0.5).mean():.0f}%")

if len(male) > 0:
    print()
    print("=== Male descriptive (n≈12 per sample, very noisy — descriptive only) ===")
    for ch_name in male["channel_name"].unique():
        aucs = male.loc[male["channel_name"] == ch_name, "auc"].dropna()
        if len(aucs) > 0:
            print(f"  {ch_name:20s}  median={aucs.median():.3f}  "
                  f"IQR=[{aucs.quantile(0.25):.3f}, {aucs.quantile(0.75):.3f}]  "
                  f">0.5: {100*(aucs>0.5).mean():.0f}%")


In [ ]:
channels = fem["channel_name"].unique()
fig, ax = plt.subplots(figsize=(7, 5))

positions = np.arange(len(channels))
data = [fem.loc[fem["channel_name"] == ch, "auc"].dropna().values for ch in channels]
vp = ax.violinplot(data, positions=positions, showmedians=True, showextrema=False)
for body in vp["bodies"]:
    body.set_facecolor("#E91E63")
    body.set_alpha(0.4)
vp["cmedians"].set_color("black")

# Overlay boxplot
bp = ax.boxplot(data, positions=positions, widths=0.15, patch_artist=True,
                showfliers=False)
for patch in bp["boxes"]:
    patch.set_facecolor("#F8BBD0")
    patch.set_edgecolor("black")

ax.axhline(0.5, color="grey", ls="--", lw=1, label="Chance (0.5)")
ax.set_xticks(positions)
ax.set_xticklabels(channels, rotation=20, ha="right")
ax.set_ylabel("LOOCV AUC (balanced 11 vs 11)")
ax.set_title("Target D — Female Sex-Matched Subsampling\n"
             "(200 reps, LOOCV) — descriptive sensitivity, not definitive falsification")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(0.0, 1.0)
fig.tight_layout()
fig.savefig(FIG / "fig9_sex_matched_subsampling.png", bbox_inches="tight")
fig.savefig(FIG / "fig9_sex_matched_subsampling.pdf", bbox_inches="tight")
plt.show()
print("Fig 9 saved.")


---
## §5  Metadata decomposition  (Target D)

Is the Age+Sex metadata AUC driven primarily by sex or by age?

This matters because the sex imbalance (χ² p = 0.025) is the most
plausible metadata confound for Target D.  If sex alone explains most
of the metadata AUC, and if sex is also a likely driver of the
connectome signal (see §4), then both the metadata model and the
connectome model may be capturing sex differences rather than fatigue biology.

Three models (same 5-fold CV):
- **Age only** — logistic regression on Age alone
- **Sex only** — logistic regression on Sex alone  
- **Age + Sex** — logistic regression on both

Permutation p-value vs. chance and bootstrap 95% CI are reported for each.


In [ ]:
meta_d = _load_json("hardening_D_metadata_decomposition.json")

if meta_d:
    print("=== Metadata Decomposition — Target D ===")
    print()
    for name in ["age_only", "sex_only", "age_sex"]:
        r = meta_d.get(name, {})
        if r:
            sig = "*" if r["perm_p"] < 0.05 else "n.s."
            print(f"  {name:12s}  AUC={r['auc']:.4f}  "
                  f"95%CI [{r['ci_95'][0]:.4f}, {r['ci_95'][1]:.4f}]  "
                  f"perm_p={r['perm_p']:.4f} ({sig})")
    print()
    # Identify sex as main driver
    auc_sex = meta_d.get("sex_only", {}).get("auc", 0)
    auc_age = meta_d.get("age_only", {}).get("auc", 0)
    auc_as  = meta_d.get("age_sex",  {}).get("auc", 0)
    if auc_sex > auc_age:
        print(f"  → Sex (AUC={auc_sex:.4f}) > Age (AUC={auc_age:.4f}): sex is the dominant metadata predictor.")
        print(f"    Adding Age to Sex raises AUC to {auc_as:.4f} — marginal gain.")
    else:
        print(f"  → Age (AUC={auc_age:.4f}) ≥ Sex (AUC={auc_sex:.4f}): age may be the dominant predictor.")
else:
    print("Metadata decomposition file not found. Rerun run_fatigue_hardening.py.")


In [ ]:
if meta_d:
    names  = ["age_only", "sex_only", "age_sex"]
    labels = ["Age only", "Sex only", "Age + Sex"]
    aucs   = [meta_d[n]["auc"] for n in names]
    ci_lo  = [meta_d[n]["ci_95"][0] for n in names]
    ci_hi  = [meta_d[n]["ci_95"][1] for n in names]
    pvals  = [meta_d[n]["perm_p"] for n in names]

    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(names))
    bars = ax.bar(x, aucs,
                  yerr=[np.array(aucs)-np.array(ci_lo), np.array(ci_hi)-np.array(aucs)],
                  capsize=5, color=["#78909C", "#E91E63", "#42A5F5"],
                  edgecolor="black", linewidth=0.5)
    ax.axhline(0.5, color="grey", ls=":", lw=0.8, label="Chance")
    for i, (b, p, a) in enumerate(zip(bars, pvals, aucs)):
        sig = "*" if p < 0.05 else ""
        ax.text(b.get_x() + b.get_width()/2, a + (np.array(ci_hi)-np.array(aucs))[i] + 0.02,
                f"{a:.3f}{sig}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("OOF AUC (5-fold CV)")
    ax.set_ylim(0.3, 0.85)
    ax.set_title("Target D — Metadata Decomposition\n"
                 "Error bars = 95% bootstrap CI; * = perm p < 0.05")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / "fig10_metadata_decomposition.png", bbox_inches="tight")
    fig.savefig(FIG / "fig10_metadata_decomposition.pdf", bbox_inches="tight")
    plt.show()
    print("Fig 10 saved.")


---
## §6  Power / precision analysis  (Target D)

The negative results in §3 (incremental value) and §4 (female-only) are
interpreted as **underpowered**, not as absent effects.

This section quantifies:
- **Observed AUC** and bootstrap standard error
- **Approximate power** at the current n = 53 (one-sided α = 0.05)
- **Rough n for 80% power** (analytical approximation, SE ∝ 1/√n)

**Method:** SE(AUC) estimated from 2 000 bootstrap resamples.
Power approximated as P(Z > z₀.₀₅ − effect/SE) where effect = AUC − 0.5.
These are conservative approximations; treat n-for-80% as order-of-magnitude.

| Negative finding | Current power (approx) | Implication |
|-----------------|------------------------|-------------|
| Incremental value n.s. | Low (< 50%) | Absence of evidence ≠ evidence of absence |
| Female-only near-chance | Very low (< 30%) | n too small for within-sex confirmation |


In [ ]:
power_d = _load_json("hardening_D_power_analysis.json")

if power_d:
    print("=== Power Analysis — Target D ===")
    print()
    print(f"  {'Model':<30s}  {'AUC':>6s}  {'SE':>6s}  {'Effect':>7s}  "
          f"{'Power@n':>8s}  {'n for 80%':>10s}")
    print("-" * 80)
    order = ["best_resid_connectome", "metadata_age_sex", "sex_only", "age_only", "combined"]
    labels = {
        "best_resid_connectome": "Best resid connectome",
        "metadata_age_sex":      "Metadata (Age+Sex)",
        "sex_only":              "Sex only",
        "age_only":              "Age only",
        "combined":              "Combined (conn+meta)",
    }
    for key in order:
        if key not in power_d:
            continue
        r = power_d[key]
        n80 = r.get("n_for_80pct_power_approx")
        n80_str = str(n80) if n80 is not None else "—"
        print(f"  {labels.get(key, key):<30s}  {r['auc']:>6.4f}  {r['bootstrap_se']:>6.4f}  "
              f"{r['effect_auc_minus_chance']:>7.4f}  {r['power_at_current_n']:>8.3f}  "
              f"{n80_str:>10s}")
    print()
    print("Note: Power at current n < 0.5 means negative result is underpowered.")
    print("Note: n-for-80% is approximate (SE ∝ 1/√n scaling).")
else:
    print("Power analysis file not found. Rerun run_fatigue_hardening.py.")


In [ ]:
if power_d:
    order  = ["best_resid_connectome", "metadata_age_sex", "sex_only", "combined"]
    labels_plot = ["Best resid\nconnectome", "Metadata\n(Age+Sex)", "Sex only", "Combined"]
    aucs_p = [power_d[k]["auc"]              for k in order if k in power_d]
    ses_p  = [power_d[k]["bootstrap_se"]     for k in order if k in power_d]
    pwrs_p = [power_d[k]["power_at_current_n"] for k in order if k in power_d]
    lbs_p  = [labels_plot[i] for i, k in enumerate(order) if k in power_d]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # AUC with SE bars
    ax = axes[0]
    x  = np.arange(len(aucs_p))
    ax.bar(x, aucs_p, yerr=ses_p, capsize=5, color="#42A5F5", edgecolor="black", linewidth=0.5)
    ax.axhline(0.5, color="grey", ls=":", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(lbs_p, fontsize=9)
    ax.set_ylabel("OOF AUC ± bootstrap SE")
    ax.set_title("Target D — AUC ± SE (n=53)")
    ax.set_ylim(0.4, 0.85)

    # Power
    ax = axes[1]
    colors_pw = ["#66BB6A" if p >= 0.5 else "#EF5350" for p in pwrs_p]
    ax.bar(x, pwrs_p, color=colors_pw, edgecolor="black", linewidth=0.5)
    ax.axhline(0.80, color="green", ls="--", lw=1.2, label="80% power target")
    ax.axhline(0.50, color="orange", ls="--", lw=1.0, label="50% power")
    ax.set_xticks(x); ax.set_xticklabels(lbs_p, fontsize=9)
    ax.set_ylabel("Approx. power (α=0.05, one-sided)")
    ax.set_title("Approximate Power at Current n=53")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

    fig.suptitle("Power / Precision Analysis — Target D", fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG / "fig11_power_analysis.png", bbox_inches="tight")
    fig.savefig(FIG / "fig11_power_analysis.pdf", bbox_inches="tight")
    plt.show()
    print("Fig 11 saved.")


---
## §7  Hardened Verdict

### Target D  (COVID-only, FATIGA EXTREMA vs NO HAY FATIGA)

| Test | Result | Interpretation |
|------|--------|---------------|
| Max-perm (within-fold PCA, resid) | corrected p = 0.028 (best ch = dFC\_AbsDiffMean, AUC=0.7284) | ✅ PASS (FWER) |
| Holm–Bonferroni (FWER, per-channel) | MI\_KNN p=0.070, dFC\_AbsDiffMean p=0.070 | ❌ Does **not** reach α=0.05 (max-perm passes, single-channel Holm does not) |
| Benjamini–Hochberg (FDR) | MI\_KNN q≈0.038, dFC\_AbsDiffMean q≈0.038 (rank-coupled) | FDR survives — exploratory only |
| Incremental value over Age+Sex | ΔAUC ≈ +0.09, p ≈ 0.20, CI crosses 0 | Not significant; power-limited |
| Female sex-matched | median AUC ≈ 0.50–0.55 | Near chance; descriptive only, low power |
| Sex as metadata driver | Sex-only AUC > Age-only AUC | Sex is dominant metadata confound |

**Summary:**  The Target D signal is **fragile and entangled with sex confounding**.
- Survives BH-FDR but not Holm-FWER.
- The paired BH q ≈ 0.038 for two channels is a **rank-coupled joint outcome**,
  not two independent discoveries.
- Does not add significant value beyond sex + age alone.
- Collapses when sex imbalance is removed, though this result is severely underpowered.
- Dynamic / nonlinear channels (MI\_KNN, dFC\_AbsDiffMean) show the most
  suggestive signal; linear static channels look weaker.  This is an
  interesting exploratory biological pointer, but remains non-confirmatory.

### Target E  (all groups, shortcut audit — not a fatigue classifier)

| Test | Result | Interpretation |
|------|--------|---------------|
| Max-perm corrected (within-fold) | raw p=0.019, resid p=0.009 (best: DistanceCorr AUC=0.707) | ✅ Strong signal |
| Group confound | χ²=18.19, p<0.0001 | **Massive COVID/CONTROL shortcut threat** |

**Summary:**  Target E shows a strong signal almost certainly driven by
the COVID vs CONTROL group composition imbalance.  It is not interpretable
as fatigue evidence.

### Overall Hardened Verdict

> ### INDETERMINATE — no robust confirmatory evidence for a fatigue-connectome signal
>
> The most charitable reading: there is an **exploratory suggestion** of
> fatigue-related information in dynamic/nonlinear FC channels (MI\_KNN,
> dFC\_AbsDiffMean) that survives FDR correction.  However:
> - It does not survive FWER correction (Holm or max-perm).
> - The FDR result is rank-coupled, not independently confirmed.
> - It does not add significant value beyond Age+Sex alone.
> - It is entangled with sex imbalance; within-sex replication is low-power.
> - Current n ≈ 53 yields very low power for the incremental-value and
>   female-only tests; negative results there are uninformative, not exculpatory.
>
> **Do NOT** claim a robust fatigue-connectome biomarker.
> **Do NOT** proceed directly to fatigue-specific representation learning
> on this evidence alone.
> Frame any publication as **exploratory / hypothesis-generating**.
>
> This verdict is aligned with `04_a` (INDETERMINATE).


In [ ]:
from IPython.display import display, HTML

display(HTML(
    "<div style='background:#f8d7da;border:2px solid #842029;padding:16px;"
    "border-radius:8px;margin:12px 0;text-align:center'>"
    "<h2 style='color:#842029;margin:0 0 8px 0'>HARDENED VERDICT</h2>"
    "<h3 style='color:#842029;margin:0'>INDETERMINATE</h3>"
    "<p style='margin:8px 0 0 0'>"
    "No robust confirmatory evidence for a fatigue–connectome signal at this stage.<br>"
    "Exploratory FDR signal in dynamic channels (MI_KNN, dFC_AbsDiffMean) does not "
    "survive FWER correction and is entangled with sex confounding.<br>"
    "Result aligned with <code>04_a</code> verdict."
    "</p></div>"
))
